In [1]:
import pandas as pd
import numpy as np

print("🚀 Initializing Time-Series Transformation...")

# 1. Load the original dataset
df = pd.read_csv('../data/aqi_dataset.csv')

# Clean missing values and engineer our custom feature
df.fillna(df.median(), inplace=True)
df['stagnation_index'] = df['industrial_activity'] / (df['wind_speed'] + 0.1)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(), inplace=True)

# 2. THE TIME-SERIES UPGRADE (Lag Features)
# We create a new column that looks exactly 1 row (hour) into the past
df['AQI_1_Hour_Ago'] = df['AQI'].shift(1)

# We can even add a 24-hour lag so the AI knows what happened yesterday at this exact time!
df['AQI_24_Hours_Ago'] = df['AQI'].shift(24)

# 3. Clean up the "Time Travel" NaN values
# The first row doesn't have a "previous hour", so it becomes NaN. We must drop these.
df.dropna(inplace=True)

# 4. Reorder columns so our Target (AQI) is at the very end (Best practice for Neural Networks)
target = df.pop('AQI')
df['AQI_Target'] = target

print("✅ Data successfully transformed into Time-Series format!")
print("\n--- New Dataset Preview ---")
print(df[['temperature', 'AQI_1_Hour_Ago', 'AQI_24_Hours_Ago', 'AQI_Target']].head())

# Save this new hyper-advanced dataset for our Neural Network
df.to_csv('../data/aqi_timeseries_dataset.csv', index=False)
print("\n💾 Saved as 'aqi_timeseries_dataset.csv'")

🚀 Initializing Time-Series Transformation...
✅ Data successfully transformed into Time-Series format!

--- New Dataset Preview ---
    temperature  AQI_1_Hour_Ago  AQI_24_Hours_Ago  AQI_Target
24        21.19          144.84            149.46      136.33
25        25.78          136.33            164.90      119.11
26        25.00          119.11            155.39      160.73
27        27.63          160.73            105.21      153.16
28        20.80          153.16            118.21      120.55

💾 Saved as 'aqi_timeseries_dataset.csv'


In [5]:
import joblib
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 1. Load the same dataset you used for training
df = pd.read_csv('../data/aqi_timeseries_dataset.csv')
X = df.drop('AQI_Target', axis=1).values
y = df['AQI_Target'].values

# 2. Re-create the scalers
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

# 3. "Fit" them to the data (this teaches them the Min/Max values)
scaler_x.fit(X)
scaler_y.fit(y.reshape(-1, 1))

# 4. Save them to the models folder
joblib.dump(scaler_x, '../models/scaler_x.pkl')
joblib.dump(scaler_y, '../models/scaler_y.pkl')

print("✅ Success! 'scaler_x.pkl' and 'scaler_y.pkl' created in models/ folder.")

✅ Success! 'scaler_x.pkl' and 'scaler_y.pkl' created in models/ folder.
